# Collect best-performing synthetic models for the paper table

Synthetic-data equivalent of `paper-collect_results_for_expanded_main_table.ipynb`.

Iterates over all targets and the `-conf.csv` files produced by the
bootstrapping notebook, selects the best-performing eGeMAPS and wav2vec2
model for each target, and collects them into a single CSV suitable
for `evaluate_session_level_ccc.py`.

**Input:** `results/synthetic/composed/everything/results-{target}-conf.csv`  
**Output:** `results/synthetic/composed/compiled-synthetic-paper.csv`

In [1]:
import os
import pandas as pd
import re

path_results_base = "../../results/synthetic/composed"
dir_composed = os.path.join(path_results_base, "everything")

file_out = "compiled-synthetic-paper"

# Targets present in the synthetic results
dct_targets = {
    "WHO-5": "who_5_percentage_score_corrected",
    "PSS-10": "pss_10_total_score",
    "PHQ-8": "phq_8_total_score",
    "Stress-now": "stress_current",
    "Stress-work": "stress_work_tasks",
}

In [2]:
# Feature sets to distinguish
dct_features = {"eGeMAPS": "eGeMAPSv02", "W2V2": "wav2vec2-variant"}

df_best = pd.DataFrame()

for cur_target_display, cur_target_val in dct_targets.items():
    conf_csv = os.path.join(dir_composed, f"results-{cur_target_val}-conf.csv")
    if not os.path.exists(conf_csv):
        print(f"WARNING: {conf_csv} not found -- skipping {cur_target_display}")
        continue

    cur_df = pd.read_csv(conf_csv)

    for cur_feat_label, cur_feat_pattern in dct_features.items():
        # Filter rows matching the feature pattern in the path
        mask = cur_df["path"].str.contains(cur_feat_pattern, na=False)
        cur_df_filter = cur_df[mask]

        if cur_df_filter.empty:
            print(
                f"WARNING: No matching rows for {cur_target_display}; "
                f"{cur_feat_label}"
            )
            continue

        # Best model: highest ccc_conf_mean
        cur_best_row = cur_df_filter.sort_values(
            by="ccc_conf_mean", ascending=False
        ).iloc[[0]]

        # Prepare columns expected by evaluate_session_level_ccc.py
        cur_best_row = cur_best_row.copy()
        cur_best_row["Target"] = cur_target_val
        cur_best_row["Task"] = cur_best_row["path"].str.split("/").str[3]
        cur_best_row.rename(columns={"features": "Features"}, inplace=True)
        # Survey label (synthetic data has no denoised/noisy distinction)
        cur_best_row["Survey"] = "synthetic-noisy"

        df_best = pd.concat([df_best, cur_best_row], ignore_index=True)

print(f"Collected {len(df_best)} best-model rows.")
df_best[
    ["Target", "Features", "Task", "ccc_conf_mean", "ccc_conf_low", "ccc_conf_high"]
]

Collected 10 best-model rows.


,Target,Features,Task,ccc_conf_mean,ccc_conf_low,ccc_conf_high
0,who_5_percentage_score_corrected,eGeMAPSv02,speechtasks-standardized_tasks,0.017421,-0.021716,0.054670
1,who_5_percentage_score_corrected,wav2vec2-variant-wav2vec2-large-robust-12-ft-e...,speechtasks-standardized_tasks,0.032685,-0.011144,0.076176
2,pss_10_total_score,eGeMAPSv02,speechtasks-standardized_tasks,-0.008479,-0.050181,0.032561
3,pss_10_total_score,wav2vec2-variant-wav2vec2-large-robust-12-ft-e...,speechtasks-standardized_tasks,-0.002647,-0.038780,0.037450
4,phq_8_total_score,eGeMAPSv02,speechtasks-standardized_tasks,-0.014538,-0.042767,0.011273
5,phq_8_total_score,wav2vec2-variant-wav2vec2-large-robust-12-ft-e...,speechtasks-standardized_tasks,-0.012139,-0.052937,0.033432
6,stress_current,eGeMAPSv02,speechtasks-standardized_tasks,-0.021311,-0.046008,0.006552
7,stress_current,wav2vec2-variant-wav2vec2-large-robust-12-ft-e...,speechtasks-standardized_tasks,0.000390,-0.040529,0.044871
8,stress_work_tasks,eGeMAPSv02,speechtasks-standardized_tasks,-0.000209,-0.036044,0.036350
9,stress_work_tasks,wav2vec2-variant-wav2vec2-large-robust-12-ft-e...,speechtasks-standardized_tasks,-0.031136,-0.062707,-0.001944


In [3]:
out_path = os.path.join(path_results_base, file_out + ".csv")
df_best.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to ../../results/synthetic/composed/compiled-synthetic-paper.csv
